# TF-IDF + Support Vector Machine (SVM)

This notebook trains a Support Vector Machine baseline for AI-generated text detection.

TF-IDF is used to convert text into numerical features, and LinearSVC is used for binary classification.

The same train, validation, and test splits used for Logistic Regression are reused for a fair comparison.

## Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import mlflow
import mlflow.sklearn

## Paths

In [2]:
DATA_DIR = Path("../data/splits")
MODEL_DIR = Path("../models")
OUTPUT_DIR = Path("../outputs")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.parquet"
VAL_FILE = DATA_DIR / "validation.parquet"
TEST_FILE = DATA_DIR / "test.parquet"

## MLflow setup

In [3]:
MLFLOW_DIR = Path("../mlruns").resolve()

mlflow.set_tracking_uri(
    f"file:///{MLFLOW_DIR.as_posix()}"
)

mlflow.set_experiment("AI Text Detection")

print("MLflow tracking URI:")
print(mlflow.get_tracking_uri())

MLflow tracking URI:
file:///C:/Users/Drasti/PROG74040-AI-Text-Detection/mlruns


C:\Users\Drasti\anaconda3\envs\aimlcourse\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## Loading datasets

In [4]:
train_df = pd.read_parquet(TRAIN_FILE)
val_df = pd.read_parquet(VAL_FILE)
test_df = pd.read_parquet(TEST_FILE)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (341052, 3)
Validation: (73083, 3)
Test: (73083, 3)


## Separate text and labels

In [5]:
X_train = train_df["text"]
y_train = train_df["generated"]

X_val = val_df["text"]
y_val = val_df["generated"]

X_test = test_df["text"]
y_test = test_df["generated"]

## TF-IDF

In [6]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=3,
    max_df=0.90
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (341052, 50000)
Validation TF-IDF shape: (73083, 50000)
Test TF-IDF shape: (73083, 50000)


## Train, evaluate, and log with ML Flow

In [7]:
if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="TF-IDF SVM"):

    # -------------------------
    # Log TF-IDF parameters
    # -------------------------

    mlflow.log_param("vectorizer", "TF-IDF")
    mlflow.log_param("max_features", 50000)
    mlflow.log_param("ngram_range", "(1, 2)")
    mlflow.log_param("stop_words", "english")
    mlflow.log_param("min_df", 3)
    mlflow.log_param("max_df", 0.90)

    # -------------------------
    # Create SVM model
    # -------------------------

    svm_model = LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=42
    )

    mlflow.log_param("model", "LinearSVC")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("class_weight", "balanced")

    # -------------------------
    # Train
    # -------------------------

    svm_model.fit(X_train_tfidf, y_train)

    print("SVM training complete.")

    # -------------------------
    # Validation
    # -------------------------

    val_preds = svm_model.predict(X_val_tfidf)
    val_scores = svm_model.decision_function(X_val_tfidf)

    val_accuracy = accuracy_score(y_val, val_preds)
    val_precision = precision_score(y_val, val_preds)
    val_recall = recall_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_preds)
    val_roc_auc = roc_auc_score(y_val, val_scores)

    print("\nValidation Results")
    print("----------------------")
    print("Accuracy:", val_accuracy)
    print("Precision:", val_precision)
    print("Recall:", val_recall)
    print("F1:", val_f1)
    print("ROC-AUC:", val_roc_auc)

    mlflow.log_metric("val_accuracy", val_accuracy)
    mlflow.log_metric("val_precision", val_precision)
    mlflow.log_metric("val_recall", val_recall)
    mlflow.log_metric("val_f1", val_f1)
    mlflow.log_metric("val_roc_auc", val_roc_auc)

    print("\nValidation Classification Report:")
    print(
        classification_report(
            y_val,
            val_preds,
            target_names=["Human", "AI"]
        )
    )

    # -------------------------
    # Test
    # -------------------------

    test_preds = svm_model.predict(X_test_tfidf)
    test_scores = svm_model.decision_function(X_test_tfidf)

    test_results = {
        "accuracy": accuracy_score(y_test, test_preds),
        "precision": precision_score(y_test, test_preds),
        "recall": recall_score(y_test, test_preds),
        "f1_score": f1_score(y_test, test_preds),
        "roc_auc": roc_auc_score(y_test, test_scores)
    }

    print("\nTest Results")
    print("----------------------")

    for metric, value in test_results.items():
        print(f"{metric}: {value}")

    mlflow.log_metric("test_accuracy", test_results["accuracy"])
    mlflow.log_metric("test_precision", test_results["precision"])
    mlflow.log_metric("test_recall", test_results["recall"])
    mlflow.log_metric("test_f1", test_results["f1_score"])
    mlflow.log_metric("test_roc_auc", test_results["roc_auc"])

    # -------------------------
    # Save model + vectorizer
    # -------------------------

    svm_model_path = MODEL_DIR / "tfidf_svm_model.joblib"
    svm_vectorizer_path = MODEL_DIR / "tfidf_svm_vectorizer.joblib"

    joblib.dump(svm_model, svm_model_path)
    joblib.dump(tfidf, svm_vectorizer_path)

    print("\nSVM model and vectorizer saved successfully.")

    # -------------------------
    # Log model + artifacts
    # -------------------------

    mlflow.sklearn.log_model(
        svm_model,
        "svm_model"
    )

    mlflow.log_artifact(str(svm_model_path))
    mlflow.log_artifact(str(svm_vectorizer_path))

    # -------------------------
    # Save results CSV
    # -------------------------

    results_df = pd.DataFrame([{
        "model": "TF-IDF + SVM",
        "accuracy": test_results["accuracy"],
        "precision": test_results["precision"],
        "recall": test_results["recall"],
        "f1_score": test_results["f1_score"],
        "roc_auc": test_results["roc_auc"]
    }])

    results_path = OUTPUT_DIR / "tfidf_svm_results.csv"

    results_df.to_csv(
        results_path,
        index=False
    )

    mlflow.log_artifact(str(results_path))

SVM training complete.

Validation Results
----------------------
Accuracy: 0.9993979448024849
Precision: 0.9993016759776536
Recall: 0.9990813551848313
F1: 0.9991915034361104
ROC-AUC: 0.9999946726651805

Validation Classification Report:
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00     45869
          AI       1.00      1.00      1.00     27214

    accuracy                           1.00     73083
   macro avg       1.00      1.00      1.00     73083
weighted avg       1.00      1.00      1.00     73083


Test Results
----------------------
accuracy: 0.9993705786571433
precision: 0.9991915034361104
recall: 0.9991180685701687
f1_score: 0.9991547846538292
roc_auc: 0.9999943293078029


2026/08/10 17:35:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



SVM model and vectorizer saved successfully.


2026/08/10 17:35:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## Show final resulat

In [8]:
results_df

,model,accuracy,precision,recall,f1_score,roc_auc
0,TF-IDF + SVM,0.999371,0.999192,0.999118,0.999155,0.999994


## VErifying MLflow contains both runs

In [9]:
runs = mlflow.search_runs(
    experiment_names=["AI Text Detection"]
)

runs[[
    "tags.mlflow.runName",
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.test_f1",
    "metrics.test_roc_auc"
]]

,tags.mlflow.runName,metrics.test_accuracy,metrics.test_precision,metrics.test_recall,metrics.test_f1,metrics.test_roc_auc
0,TF-IDF SVM,0.999371,0.999192,0.999118,0.999155,0.999994
1,TF-IDF Logistic Regression,0.994992,0.993312,0.993239,0.993275,0.999743
